In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.rcParams["font.family"] = "Times New Roman"

# Load data
file_path = "your_path/src/tasks/pretrain/visualization/plots/teacher에 따른 성능(AUROC,AUPRC).xlsx"
df_auroc = pd.read_excel(file_path, sheet_name="AUROC")
df_aprc = pd.read_excel(file_path, sheet_name="AUPRC")
df_auroc.rename(columns={df_auroc.columns[0]: "Teacher"}, inplace=True)
df_aprc.rename(columns={df_aprc.columns[0]: "Teacher"}, inplace=True)

# normalization 범위
norm_min, norm_max = 40, 80


def normalize(df, mn=norm_min, mx=norm_max):
    dfn = df.copy()
    for c in df.columns[1:]:
        lo, hi = df[c].min(), df[c].max()
        dfn[c] = mn + (df[c] - lo) / (hi - lo) * (mx - mn)
    return dfn


df_auroc_n = normalize(df_auroc)
df_aprc_n = normalize(df_aprc)

styles = ["-", "--", "-.", ":"]
data_colors = ["#C5172E", "#537D5D", "#4E71FF", "#E9A319", "#94B4C1"]
fill_alphas = [0.15] * 4

for df_raw, df_norm, metric in [(df_auroc, df_auroc_n, "AUROC"), (df_aprc, df_aprc_n, "AUPRC")]:

    cats = df_norm.columns[1:].tolist()
    raw_min = {c: df_raw[c].min() for c in cats}
    raw_max = {c: df_raw[c].max() for c in cats}

    angles = np.linspace(0, 2 * np.pi, len(cats), endpoint=False).tolist()
    angles += angles[:1]

    fig, ax = plt.subplots(figsize=(6, 6), subplot_kw={"polar": True})
    ax.spines["polar"].set_visible(False)
    ax.set_ylim(0, 100)
    ax.yaxis.grid(False)
    ax.set_xticks([])
    ax.set_yticks([])

    theta = np.linspace(0, 2 * np.pi, 200)
    ax.plot(theta, [norm_min] * 200, color="#D7D7D7", linestyle="-", linewidth=0.8, zorder=0)
    ax.plot(theta, [norm_max] * 200, color="#D7D7D7", linestyle="-", linewidth=0.8, zorder=0)

    # spokes
    for ang in angles[:-1]:
        ax.plot([ang, ang], [0, 100], color="#D7D7D7", linestyle="-", lw=0.8, zorder=0)

    # performance curves + fills
    for i, row in enumerate(df_norm.itertuples(index=False)):
        color = data_colors[i % len(data_colors)]
        vals = list(row)[1:] + [list(row)[1]]
        ax.plot(
            angles,
            vals,
            linestyle=styles[i],
            lw=2.5 if i == 3 else 2,
            label=row.Teacher,
            color=color,
        )
        ax.fill(angles, vals, alpha=fill_alphas[i], color=color)

    # 최소/최대 값 텍스트만 찍기 (마커 제거)
    for ang, cat in zip(angles[:-1], cats):
        # 최소값
        ax.text(ang, norm_min + 7.5, f"{raw_min[cat]:.1f}", ha="center", va="center", fontsize=12)
        # 최대값
        ax.text(ang, norm_max + 7.5, f"{raw_max[cat]:.1f}", ha="center", va="center", fontsize=12)

    # dataset labels
    label_r = 105
    for ang, text in zip(angles[:-1], cats):
        words = text.split(" ")
        m = len(words) // 2 or 1
        lbl = f"{' '.join(words[:m])}\n{' '.join(words[m:])}"
        rot = np.degrees(ang) - 90
        if rot < -90:
            rot += 180

        # 색상 결정
        if text.startswith("Diabetic retinopathy"):
            col = "red"
        elif text.startswith("Glaucoma"):
            col = "green"
        elif text.startswith("Age-related macular degeneration"):
            col = "blue"
        else:
            col = "black"

        ax.text(
            ang,
            label_r,
            lbl,
            rotation=rot,
            rotation_mode="anchor",
            ha="center",
            va="bottom",
            multialignment="center",
            fontfamily="Times New Roman",
            color=col,  # ← 텍스트 컬러
            fontsize=12,
        )

    ax.legend(
        loc="upper right",
        bbox_to_anchor=(1.3, 1.1),
        prop={"family": "Times New Roman", "size": 11},
    )
    plt.show()